## Error differences

This section is about analysing the differences made to errors by reconciliation of the four hierarchies.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy, BinaryTemporalHierarchy, FullTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/binary_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
bth = BinaryTemporalHierarchy()
fth = FullTemporalHierarchy()

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=fth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=bth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
])

nrmse_em, nrmse_diffs = em, diffs

### Difference plots

Here we look at plots visualising the difference made to base error values by reconciliation per hierarchy and granularity. First we look at monthly granularity.

In [ ]:
for h in (mth, sth, bth, fth):
    plots.plot_error_diff(nrmse_em, MI(forecast_type=BASE_FORECASTS, granularity=MONTHS),
                                    MI(forecast_type=RECONCILED_FORECASTS, hierarchy=h, granularity=MONTHS),
                                    outlier_threshold=0.99).show()

There is not a big visually dinstiction between the hierarchy performances. Next we look at daily granularity errors.

In [ ]:
for h in (mth, sth, bth, fth):
    plots.plot_error_diff(nrmse_em, MI(forecast_type=BASE_FORECASTS, granularity=DAYS),
                                    MI(forecast_type=RECONCILED_FORECASTS, hierarchy=h, granularity=DAYS),
                                    outlier_threshold=0.99).show()

Visually we can tell, that with `Binary` and `Full` hierarchy, the big base errors are are very rarely increased (and even then it's a minor increase), and are frequently decreased by a lot, especially compared to `Minimal`.

### Error sign distributions

Next I look at the distributions of the signs of differences at daily and monthly granularity.

In [ ]:
for h in (mth, sth, bth, fth):
    plots.plot_error_signs(nrmse_diffs, MI(granularity=MONTHS, hierarchy=f'{h} - Minimal'), MI(granularity=DAYS, hierarchy=f'{h} - Minimal'), percentage=True).show()

By a little margin, `Full` has the most amount of negative-negative difference signs, but also the most amount of positive-positive changes. This hierarchy also has the most amount of daily positive changes (60%). Monthly positive changes on the other hand are consistent across the hierarchies.

### Error difference statistics

Next we look at the statistics of the error differences, both raw (with outliers filtered out) and filtered by signs. First we look at the complete statistics at monthly then daily granularity.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=MONTHS),
                            df_filter=lambda df: df[(df < np.quantile(df.values.flatten(), 0.9995)) & (df > np.quantile(df.values.flatten(), 0.0001))])

Looking at monthly error difference statistics, with outliers sorted out, `Full` has the smallest mean, smallest variance, second smallest median (with `Semantic` having the smallest). When looking at the progression of mean and variance in order of the hierarchies, a trend can be seen, where the more aggregation levels seem to provide bigger mean error decreases, and lesser variation in the changes.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=DAYS),
                            df_filter=lambda df: df[(df < np.quantile(df.values.flatten(), 0.9995)) & (df > np.quantile(df.values.flatten(), 0.0001))])

Looking at daily error difference statistics, with outliers sorted out, `Full` has the smallest mean, the highest variance, and biggest median. In terms of the mean, variance, and median, a trend can be observed between the four hierarchies: `Full` favored by lowest mean, `Minimal` favored by lowest median, and vica versa

#### Filtered statistics

Next I look at the statistics of the error differences, after sorting out only positive/negative differences at monthly and daily granularity separately.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=MONTHS), df_filter=lambda df: df[(df > 0) & (df < np.quantile(df.values.flatten(), 0.999))])

When looking at monthly positive error differences, with outliers sorted out, we can see that the counts of samples are the same between the four hierarchies. `Full` has the smallest mean and variance, both of which have a noticable trend among the hierarchies. The median values are very similar and close across the four hierarchies, but `Semantic` is the smallest. Above the 75th percentile, it seems like a trend favoring `Full` becomes visible.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=MONTHS), df_filter=lambda df: df[(df < 0) & (df > np.quantile(df.values.flatten(), 0.001))])

When looking at monthly negative error differences, with outliers sorted out, the mean and variance is very similar across the hierarchies. `Full` has smallest mean however. It also has the smallest median value, which has a visible trend.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=DAYS), df_filter=lambda df: df[(df > 0) & (df < np.quantile(df.values.flatten(), 0.999))])

When looking at daily positive error differences, with outliers sorted out, we can see that `Full` has the most amount of cases (as we already saw in the heatmap matrix), with a visible trend across the four hierarchies. `Full` also has the largest mean with a visible trend. Variance values look all over the place, with `Semantic` having the smallest. In terms of median values, there is a visible trend, with `Full` having the largest value.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=DAYS), df_filter=lambda df: df[(df < 0) & (df > np.quantile(df.values.flatten(), 0.001))])

When looking at daily negative error differences, with outliers sorted out, `Full` has the smallest mean, with a visible trend, but it also has the biggest variance, also with a visible trend. `Full` has the smallest median, with an apparent trend, which propagates through basically all the percentiles.

### Conclusions

**Full**
- Pros
    - When looking at daily error difference plots, big daily base errors were rarely increased by reconciliation, and were frequently decreased by a large amount.
    - When looking at error sign distributions, this hierarchy had the most negative-negative (both granularities improved) cases.
    - When looking at monthly error difference statistics (with outliers filtered), this hierarchy had the smallest mean and variance, and the second smallest median.
    - When looking at daily error difference statistics (with outliers filtered), this hierarchy had the smallest mean and variance.
    - When looking at monthly positive error differences, this hierarchy had the lowest mean and variance.
    - When looking at monthly negative error differences, this hierarchy had the smallest mean and median.
    - When looking at daily negative error differences, this hierarchy had the smallest mean and median.
- Cons
    - When looking at error sign distributions, this hierarchy had the most positive-positive (both granularities worsened) cases, and the most daily positive changes overall.
    - When looking at daily error difference statistics (with outliers filtered), this hierarchy had the biggest median.
    - When looking at daily positive error differences, this hierarchy had the biggest mean.
    - When looking at daily negative error differences, this hierarchy had the biggest variance.

**Likely trends**
- Improving with more levels:
    - Monthly diff mean and variance
    - Daily diff mean and variance
    - Monthly positive diff mean, variance, and q75/q90
    - Monthly negative diff median
    - Daily negative diff mean and median
- Worsening with more levels:
    - Daily diff median
    - Daily positive diff count, mean, and median
    - Daily negative diff variance

Highlights:
- Daily error difference plots
- Error difference statistics (total and filtered)